In [ ]:
#!pip install langchain-groq langchain_huggingface langchain-community langchain-chroma pypdf

In [ ]:
import os
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated, Sequence
from langchain_core.messages import BaseMessage, SystemMessage, HumanMessage, ToolMessage
from operator import add as add_messages
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.tools import tool
# Set a rotated GROQ_API_KEY outside the notebook.

In [ ]:
llm=ChatGroq(model='llama-3.3-70b-versatile',temperature=0)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

In [ ]:
pdf_files = [
    ("gs_outlook.pdf", "Goldman Sachs", "market_outlook", 2025),
    ("jpm_outlook.pdf", "JP Morgan", "market_outlook", 2025),
    ("imf_outlook.pdf", "IMF", "macro_report", 2025),
]

all_pages = []

for path, publisher, doc_type, year in pdf_files:
    loader = PyPDFLoader(path)
    pages = loader.load()

    for page in pages:
        page.metadata["source"] = path
        page.metadata["publisher"] = publisher
        page.metadata["document_type"] = doc_type
        page.metadata["year"] = year

    all_pages.extend(pages)

print(f"Loaded {len(all_pages)} total pages")

In [ ]:
text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
pages_split=text_splitter.split_documents(all_pages)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
persist_directory = "/content/drive/MyDrive/langgraph/finance_rag"
os.makedirs(persist_directory, exist_ok=True)
collection_name = "finance_multi_doc"

In [ ]:
#Create chroma db using embedding model
vectorstore = Chroma(
    persist_directory=persist_directory,
    embedding_function=embeddings,
    collection_name=collection_name
)

In [ ]:
retriever=vectorstore.as_retriever(search_type='similarity',search_kwargs={'k':6})

In [ ]:
@tool
def retriever_tool(query:str)->str:
  """
  This tool returns ONLY text retrieved from the provided documents.
  Do NOT reference any sources or reports that are not explicitly returned
  by this tool. If information is missing, say so explicitly.
  """
  docs = retriever.invoke(query)
  if not docs:
    return "I found no relevant information in the Stock Market Performance 2024 document."
  results = []
  for i, doc in enumerate(docs):
    source = doc.metadata.get("publisher", "Unknown")
    year = doc.metadata.get("year", "Unknown year")
    page = doc.metadata.get("page", "Unknown page")
    results.append(f"Source: {source} ({year}), page {page}\n{doc.page_content}")
  return "\n\n".join(results)
tools=[retriever_tool]
llm=llm.bind_tools(tools)

In [ ]:
class AgentState(TypedDict):
  messages: Annotated[Sequence[BaseMessage],add_messages]

In [ ]:
def should_continue(state:AgentState):
  """Check if the last message contains tool calls."""
  result = state['messages'][-1]
  return hasattr(result, 'tool_calls') and len(result.tool_calls) > 0

In [ ]:
system_prompt = """
You are an intelligent AI assistant that answers questions using multiple financial documents,
including investment outlooks from Goldman Sachs and JP Morgan, and macroeconomic data from the IMF.
When answering:
- Attribute claims to their source (Goldman, JP Morgan, IMF)
- Compare viewpoints when they differ
- Cite document source and year
- Prefer IMF data for factual macroeconomic statistics
IMPORTANT:
- Every factual claim MUST be followed by a citation in the format:
  (Source, Year, Page)
- Only use citations that appear in the retriever tool output.
- If a claim cannot be cited, explicitly say: "Not found in retrieved documents."
- Do NOT summarize or paraphrase without citations.
"""
tools_dict={our_tool.name:our_tool for our_tool in tools}

In [ ]:
def call_llm(state:AgentState)->AgentState:
  """Function to call the LLM with the current state"""
  messages=list(state['messages'])
  messages=[SystemMessage(content=system_prompt)]+messages
  message=llm.invoke(messages)
  return {'messages':[message]}
def take_action(state: AgentState) -> AgentState:
  """Execute tool calls from the LLM's response."""

  tool_calls = state['messages'][-1].tool_calls
  results = []
  for t in tool_calls:
    print(f"Calling Tool: {t['name']} with query: {t['args'].get('query', 'No query provided')}")

    if not t['name'] in tools_dict: # Checks if a valid tool is present
      print(f"\nTool: {t['name']} does not exist.")
      result = "Incorrect Tool Name, Please Retry and Select tool from List of Available tools."

    else:
      result = tools_dict[t['name']].invoke(t['args'].get('query', ''))
      print(f"Result length: {len(str(result))}")


    # Appends the Tool Message
    results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))

  print("Tools Execution Complete. Back to the model!")
  return {'messages': results}

In [ ]:
graph = StateGraph(AgentState)
graph.add_node("llm", call_llm)
graph.add_node("retriever_agent", take_action)

graph.add_conditional_edges(
    "llm",
    should_continue,
    {True: "retriever_agent", False: END}
)
graph.add_edge("retriever_agent", "llm")
graph.set_entry_point("llm")

rag_agent = graph.compile()

In [ ]:
def running_agent():
  print("\n=== RAG AGENT===")

  while True:
    user_input = input("\nWhat is your question: ")
    if user_input.lower() in ['exit', 'quit']:
      break

    messages = [HumanMessage(content=user_input)] # converts back to a HumanMessage type

    result = rag_agent.invoke({"messages": messages})

    print("\n=== ANSWER ===")
    print(result['messages'][-1].content)


running_agent()